In [8]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
import operator

In [9]:
load_dotenv()

True

In [10]:
model = ChatGroq(
    model="llama-3.1-8b-instant"  # free model
    
)

In [11]:
class EvalSchema(BaseModel):
    feedback:str=Field(description="detailed Feedback for the essay")
    score:int=Field(description="score out of 10",ge=0,le=10)

In [12]:
structuredmodel=model.with_structured_output(EvalSchema)

In [13]:
essay=""" Artificial Intelligence (AI) is transforming the world by enabling machines to think and learn like humans. It is used in healthcare, education, business, and entertainment to improve efficiency and solve complex problems. AI helps automate tasks, saves time, and increases productivity, making modern life smarter, faster, and more convenient. """

In [7]:
prompt=f"Please evaluate the following essay and provide feedback and a score out of 10.\n\nEssay:\n{essay}"
structuredmodel.invoke(prompt)

EvalSchema(feedback='The essay is well-structured and provides a clear overview of the impact of AI on various sectors. However, it lacks depth and specific examples to support the claims. Additionally, the conclusion feels abrupt. Consider expanding on the benefits and challenges of AI and providing more concrete evidence to strengthen the argument.', score=6)

In [14]:
class UPSCState(TypedDict):
    essay:str
    langfeed:str
    analysisfeed:str
    clarityfeed:str
    overallfeed:str
    #adding reducer(add) to merge lists from 3 nodes in to  a single list
    indvidualscores:Annotated[list[int], operator.add]
    avg_score:float

In [15]:
def evallang(state:UPSCState)->UPSCState:
    prompt=f"Please evaluate the following essay for language and grammar and provide feedback and a score out of 10.\n\nEssay:\n{state['essay']}"
    response=structuredmodel.invoke(prompt)
    return {'langfeed':response.feedback,'indvidualscores':[response.score]}

In [16]:
def evalanalysis(state:UPSCState)->UPSCState:
    prompt=f"Please evaluate the following essay for analysis and provide feedback and a score out of 10.\n\nEssay:\n{state['essay']}"
    response=structuredmodel.invoke(prompt)
    return {'analysisfeed':response.feedback,'indvidualscores':[response.score]}

In [17]:
def evalclarity(state:UPSCState)->UPSCState:
    prompt=f"Please evaluate the following essay for clarity and provide feedback and a score out of 10.\n\nEssay:\n{state['essay']}"
    response=structuredmodel.invoke(prompt)
    return {'clarityfeed':response.feedback,'indvidualscores':[response.score]}

In [25]:
def evaloverall(state:UPSCState)->UPSCState:
    prompt=f"Please evaluate all the feedback and provide me a overall feedback, the feedbacks are{state['langfeed']}, {state['analysisfeed']}, and {state['clarityfeed']}"
    response=structuredmodel.invoke(prompt)
    overall_score=sum(state['indvidualscores'])/len(state['indvidualscores'])
    return {'overallfeed':response.feedback,'overall_score':overall_score}

In [20]:
graph=StateGraph(UPSCState)

graph.add_node('evallang',evallang)
graph.add_node('evalanalysis',evalanalysis)
graph.add_node('evalclarity',evalclarity)
graph.add_node('evaloverall',evaloverall)

graph.add_edge(START,'evallang')
graph.add_edge(START,'evalanalysis')
graph.add_edge(START,'evalclarity')
graph.add_edge('evallang','evaloverall')
graph.add_edge('evalanalysis','evaloverall')
graph.add_edge('evalclarity','evaloverall')
graph.add_edge('evaloverall',END)

workflow=graph.compile()

In [26]:
initial_state={
    'essay':essay
}
final_state=workflow.invoke(initial_state)
print(final_state['overallfeed'])

The essay is well-structured and clearly states the impact of AI on various industries. However, it lacks specific examples and details to support its claims. Additionally, the language is simple and lacks depth. To improve, consider adding more concrete examples and using more complex vocabulary. Overall, the essay shows a good understanding of the topic but falls short in its execution.
